Embedding a query.

In [1]:
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

v = embed.encode(q)

In [3]:
print(v[0])

-0.020582036807885073


Cosine similarity.

In [9]:
from gitsource import GithubRepositoryDataReader
import numpy as np

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
doc = next(d for d in documents if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md")

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
v_q = embed.encode(q)
v_doc = embed.encode(doc["content"])

similarity = v_q.dot(v_doc)
print(similarity)

0.361070280302606


Chunking and search by hand.

In [11]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [12]:
X = embed.encode_batch([c["content"] for c in chunks])

In [13]:
scores = X.dot(v)

In [14]:
best_idx = scores.argmax()
best_chunk = chunks[best_idx]

print(best_chunk["filename"])
print(scores[best_idx])

02-vector-search/lessons/07-sqlitesearch-vector.md
0.648901732433228


Vector search with `minsearch`.

In [15]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)

results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

Text search versus vector search.

In [16]:
from minsearch import Index

text_index = Index(text_fields=["content"], keyword_fields=["course"])
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

query_vector = embed.encode(query)
vector_results = vindex.search(query_vector, num_results=5)

text_results = text_index.search(query, num_results=5)

vector_filenames = {r["filename"] for r in vector_results}
text_filenames = {r["filename"] for r in text_results}

only_in_vector = vector_filenames - text_filenames
print(only_in_vector)

{'02-vector-search/lessons/08-pgvector.md'}


Hybrid search.

In [17]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query = "How do I give the model access to tools?"

query_vector = embed.encode(query)
vector_results = vindex.search(query_vector, num_results=5)
text_results = text_index.search(query, num_results=5)

results = rrf([vector_results, text_results])

print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
